In [1]:
import Pkg
Pkg.activate("../chebcoefs")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D/chebcoefs`


In [2]:
using StaticArrays
using FastChebInterp

In [3]:
struct ChebyshevSeries{T, N}
    coefs::Array{T, N}
    lb::SVector{N, T}
    ub::SVector{N, T}
end


function ChebyshevSeries(coefs::Array{T, 0}, lb::SVector{0}, ub::SVector{0}) where T
    return coefs[]
end


"""Converts a point `x` to its normalized coordinates in ``[-1, 1]^N``."""
function normalize(f::ChebyshevSeries{T, N}, x::SVector{N, T}) where {T, N}
    @. (2x - f.lb - f.ub) / (f.ub - f.lb)
end


function normalize(f::ChebyshevSeries{T, 1}, x::T) where T
    return normalize(f, SVector{1, T}(x))[]
end


"""Checks if the point `x` is in the domain of `f`."""
function contains(f::ChebyshevSeries{T, N}, x::SVector{N, T}) where {T, N}
    return all(f.lb .≤ x .≤ f.ub)
end


function contains(f::ChebyshevSeries{T, N}, x::AbstractVector{T}) where {T, N}
    return contains(f, SVector{N, T}(x))
end


function contains(f::ChebyshevSeries{T, 1}, x::T) where T
    return contains(f, SVector{1, T}(x))
end


include("clenshaw.jl")
include("gradient.jl")
include("hessian.jl");


######################################################################################################


struct Transformation{F<:Function, G<:Function, H<:Function}
    u::F
    ∇u::G
    Hu::H
end


struct ChebyshevCluster{T, N, M}
    series::NTuple{M, ChebyshevSeries{T, N}}
    tforms::NTuple{M, Transformation}
end


# function ChebyshevCluster(series::ChebyshevSeries{T, N}...) where {T, N}
#     M = length(series)
#     return ChebyshevCluster{T, N, M}(series)
# end


function contains(g::ChebyshevCluster{T, N, M}, x::Union{AbstractVector{T}, T}) where {T, N, M}
    for i in 1:M
        u = g.tforms[i].u(x)
        if contains(g.series[i], u)
            return true, i
        end
    end
    return false, nothing
end


function (g::ChebyshevCluster{T, N, M})(x::Union{AbstractVector{T}, T}) where {T, N, M}
    for i in 1:M
        u = g.tforms[i].u(x)
        if contains(g.series[i], u)
            return g.series[i](u)
        end
    end
    throw(DomainError(x))
end


function gradient(g::ChebyshevCluster{T, N, M}, x::Union{AbstractVector{T}, T}) where {T, N, M}
    for i in 1:M
        u = g.tforms[i].u(x)
        if contains(g.series[i], u)
            f, ∇ᵤf = gradient(g.series[i], u)
            ∇u = g.tforms[i].∇u(x)
            
            ∇f = ∇ᵤf * ∇u  # ∂f/∂u ⋅ ∂u/∂x
            
            return f, ∇f
        end
    end
    throw(DomainError(x))
end


function hessian(g::ChebyshevCluster{T, N, M}, x::Union{AbstractVector{T}, T}) where {T, N, M}
    for i in 1:M
        u = g.tforms[i].u(x)
        if contains(g.series[i], u)
            f, ∇ᵤf, Hᵤf = hessian(g.series[i], u)
            ∇u = g.tforms[i].∇u(x)
            Hu = g.tforms[i].Hu(x)
            
            ∇f = ∇ᵤf * ∇u                    # ∂f/∂u ⋅ ∂u/∂x
            Hf = ∇ᵤf * Hu + ∇u * Hᵤf * ∇u    # ∂f/∂u ⋅ ∂²u/∂x² + ∂²f/∂u² ⋅ (∂u/∂x)²
            
            return f, ∇f, Hf
        end
    end
    throw(DomainError(x))
end;

In [52]:
# function f1(u)
#     x = sqrt(u)
#     return sin(x)
# end

# function ∇f1(u)
#     x = sqrt(u)
#     return cos(x)
# end

# function Hf1(u)
#     x = sqrt(u)
#     return -sin(x)
# end

# function trf1(x::Float64)
#     return x^2
# end

# function trg1(x::Float64)
#     return 2*x
# end

# function trh1(x::Float64)
#     return 2.0
# end

function f1(u)
    x = u^3
    return sin(x)
end

function ∇f1(u)
    x = u^3
    return cos(x)
end

function Hf1(u)
    x = u^3
    return -sin(x)
end

function trf1(x::Float64)
    return x^1/3
end

function trg1(x::Float64)
    return 1/3*x^(-2/3)
end

function trh1(x::Float64)
    return -2/9*x^(-5/3)
end

x1, x2 = 0.0, 0.25*π
u1, u2 = trf1(x1), trf1(x2)

xc1 = chebpoints(1000, u1, u2)
cb1 = chebinterp(f1.(xc1), u1, u2)

tf1 = Transformation(trf1, trg1, trh1)
cs1 = ChebyshevSeries(cb1.coefs, SA[u1], SA[u2]);

In [53]:
function f2(u)
    x = u^2
    return sin(x)
end

function ∇f2(u)
    x = u^2
    return cos(x)
end

function Hf2(u)
    x = u^2
    return -sin(x)
end

function trf2(x::Float64)
    return x^0.5
end

function trg2(x::Float64)
    return 0.5*x^-0.5
end

function trh2(x::Float64)
    return -0.25*x^-1.5
end

x3 = 0.5*π
v1, v2 = trf2(x2), trf2(x3)

xc2 = chebpoints(50, v1, v2)
cb2 = chebinterp(f2.(xc2), v1, v2)

tf2 = Transformation(trf2, trg2, trh2)
cs2 = ChebyshevSeries(cb2.coefs, SA[v1], SA[v2]);

In [54]:
cc = ChebyshevCluster((cs1, cs2), (tf1, tf2));

In [55]:
x̄1 = 0.5*(x1 + x2)
ū1 = x̄1^2

x̄2 = 0.5*(x2 + x3)
v̄2 = sqrt(x̄2);

In [56]:
contains(cc, x̄1)

(true, 1)

In [57]:
contains(cc, x̄2)

(true, 2)

In [60]:
cc(x̄1) - f1(ū1)

-0.0014244714933805747

In [61]:
cc(x̄2) - f2(v̄2)

-1.1102230246251565e-16

In [62]:
f̄1, ∇f̄1 = gradient(cc, x̄1)
println(f̄1, ", ", f1(ū1))
println(∇f̄1, ", ", ∇f1(ū1))
println(f̄1 - f1(ū1))
println(∇f̄1 - ∇f1(ū1))

0.0022429290135193977, 0.0036674005068999724
0.03195235792277898, 0.9999932750641486
-0.0014244714933805747
-0.9680409171413696


In [63]:
f̄2, ∇f̄2 = gradient(cc, x̄2)
println(f̄2, ", ", f2(v̄2))
println(∇f̄2, ", ", ∇f2(v̄2))
println(f̄2 - f2(v̄2))
println(∇f̄2 - ∇f2(v̄2))

0.9238795325112865, 0.9238795325112866
0.3826834323650906, 0.38268343236509
-1.1102230246251565e-16
6.106226635438361e-16


In [64]:
f̄1, ∇f̄1, Hf̄1 = hessian(cc, x̄1)
println(f̄1, ", ", f1(ū1))
println(∇f̄1, ", ", ∇f1(ū1))
println(Hf̄1, ", ", Hf1(ū1))
println(f̄1 - f1(ū1))
println(∇f̄1 - ∇f1(ū1))
println(Hf̄1 - Hf1(ū1))

0.0022429290135193977, 0.0036674005068999724
0.03195235792277898, 0.9999932750641486
0.2492125228795813, -0.0036674005068999724
-0.0014244714933805747
-0.9680409171413696
0.25287992338648124


In [65]:
f̄2, ∇f̄2, Hf̄2 = hessian(cc, x̄2)
println(f̄2, ", ", f2(v̄2))
println(∇f̄2, ", ", ∇f2(v̄2))
println(Hf̄2, ", ", Hf2(v̄2))
println(f̄2 - f2(v̄2))
println(∇f̄2 - ∇f2(v̄2))
println(Hf̄2 - Hf2(v̄2))

0.9238795325112865, 0.9238795325112866
0.3826834323650906, 0.38268343236509
-0.923879532511206, -0.9238795325112866
-1.1102230246251565e-16
6.106226635438361e-16
8.060219158778636e-14


In [4]:
a = [1]
b = [2]
c = a' * b

2

In [ ]:
# function f1(u)
#     x = sqrt(u)
#     return sin(x)
# end

# function trf1(x::SVector{1, Float64})
#     return SA[x[1]^2]
# end

# function trg1(x::SVector{1, Float64})
#     return SA[2*x[1]]
# end

# function trh1(x::SVector{1, Float64})
#     return SA[2.0]
# end

# x1, x2 = 0.0, 0.25*π
# u1, u2 = x1^2, x2^2

# xc = chebpoints(30, u1, u2)
# cb = chebinterp(f1.(xc), u1, u2)

# tf1 = Transformation(trf1, trg1, trh1)
# cs1 = ChebyshevSeries(cb.coefs, SA[u1], SA[u2]);

In [7]:
# function f(x::SVector{3, Float64})
#     return SA[x[1], x[2], log(x[3])]
# end

# function g(x::SVector{3, Float64})
#     return SA[1.0, 1.0, 1/x[3]]
# end

# function h(x::SVector{3, Float64})
#     return SA[1.0, 1.0, 1/x[3]]
# end

g (generic function with 1 method)